In [ ]:
# NOTEBOOK NAME
# SimpleCAPPIs.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

import matplotlib.colors as mcolors
import matplotlib.cm as cm

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

from matplotlib.patches import Circle # for radar range ring circles on the map
from matplotlib.lines import Line2D # for plotting stars on the map legend

In [ ]:
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
# RadarIDno = '23' 

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '08:00'
LoopEndTime   = '108:00'

# CHOOSE YOUR ALTITUDE
# Altitude = 2000  # [m] choose a multiple of 500 m to look at a CAPI for

# # CHOOSE YOUR VARIABLE
# Var = 'CC'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# CHOOSE YOUR QUALITY CONTROL SETTINGS
# taken from Aragon et al. 2024
MinValidZDR = -4 # NO VALID DATA TO USE THE OPTION YET
MaxValidZDR =  4 # NO VALID DATA TO USE THE OPTION YET
MinValidRhoHV = 0.00 # please keep 2 decimal places on here for the string printing


# USER CHOICE FOLLOW-ON SECTION

# RADAR CHOICE FOLLOW-ON
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton (Brisbane)'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marburg'
elif (RadarIDno == '19'):
    RadarSiteName = 'Cairns'
elif (RadarIDno == '8'):
    RadarSiteName = 'Gympie'
elif (RadarIDno == '24'):
    RadarSiteName = 'Bowen'
elif (RadarIDno == '23'):
    RadarSiteName = 'Gladstone'
elif (RadarIDno == '74'):
    RadarSiteName = 'Greenvale'
elif (RadarIDno == '98'):
    RadarSiteName = 'Taroom'
elif (RadarIDno == '108'):
    RadarSiteName = 'Towoomba'
elif (RadarIDno == '78'):
    RadarSiteName = 'Weipa'
elif (RadarIDno == '72'):
    RadarSiteName = 'Emerald'
elif (RadarIDno == '41'):
    RadarSiteName = 'Willis Island'
else:
    RadarSiteName = 'Site ' + RadarIDno

# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# Longitude shift correction for known coordinate errors
if RadarSiteName == 'Townsville':
    LonShift = 146.5505 - (-19.4195)
else:
    LonShift = 0.0

# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -30 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 10

elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -5.0
    VarColourBar_max = 5.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0

elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.8
    VarColourBar_max = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 0.05
    
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.0
    VarColourBar_max = 10.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0

elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.0
    VarColourBar_max = 30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max =  30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'Horz'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
    #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


    RadarGridsFolder = 'CompressedRadarGrids'
    NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                             + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
    # try to load in the netcdf file and if it doesn't work, just keep going through the loop
    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # index in the netcdf altitude variable for the altitude you want
    alti = np.where(xgrid.z == Altitude) # this is a double nested array for some reason
    alti = alti[0][0] # take the index out of the double nested array

    # quit out if the altitude does not correspond to one in the netCDF file
    if ( np.size(alti) != 1): 
        raise ValueError( str(Altitude) + ' m is not a valid altitude in the data')


    # ROUGH QUALITY CONTROL SECTION
    # create a mask only where these quality control condtions are met
    # ConditionGridA = xgrid['corrected_differential_reflectivity'] > MinValidZDR   # (y, x) boolean masks
    
    # I WOULD LIKE TO ADD CONDITIONS WITH DIFFERENTIAL REFLECTIVITY, BUT THIS RADAR DOES NOT HAVE VALID DATA YET
    # ConditionGridB = xgrid['corrected_differential_reflectivity'] < MaxValidZDR  
    # ConditionGridC = xgrid['corrected_cross_correlation_ratio'] > MinValidRhoHV
    
    # ConditionGrid = ConditionGridA #* ConditionGridB * ConditionGridC   # combined boolean mask



    fig, ax = plt.subplots(figsize=(8, 6), 
                           subplot_kw={'projection': ccrs.PlateCarree()})

     # gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    
    # TERRAIN SHADING USING LOCAL GEBCO DEM
    lon_min, lon_max = float(xgrid.lon.min()) + LonShift, float(xgrid.lon.max()) + LonShift
    lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

    try:
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        
        # print(f"Attempting to load DEM with bounds:")
        # print(f"  Lon: {lon_min} to {lon_max}")
        # print(f"  Lat: {lat_min} to {lat_max}")
        
        dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)
        
        if dem_da is None:
            raise ValueError('GEBCO DEM returned None')
        
        dem_lon = dem_da.lon.values
        dem_lat = dem_da.lat.values
        dem_data = dem_da.values
  
        # print(f"DEM data range: {np.nanmin(dem_data)} to {np.nanmax(dem_data)} m")
        # print(f"NaN count: {np.isnan(dem_data).sum()}")

        # Make 2D lon/lat grids if necessary
        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        # Ensure we have some valid data
        valid = np.isfinite(dem_data)
        if not np.any(valid):
            raise ValueError('DEM has no finite values in this domain')

        colours = [
            '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
        
            '#c4dec2',  # 1: 0–200 m, pale green
            '#e4edc9',  # 2: 200–400 m, greenish-yellow
            '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
            '#e9d7bd',  # 4: 600–800 m, light tan
            '#ddc4aa',  # 5: 800–1000 m, tan
            '#cfb194',  # 6: 1000–1200 m, light brown
            '#b58f6e',  # 7: > 1200 m, darker brown
        ]
        
        bounds = [
            -1000.0,  # ocean below 0
            0.0,      # 0–200
            200.0,    # 200–400
            400.0,    # 400–600
            600.0,    # 600–800
            800.0,    # 800–1000
            1000.0,   # 1000–1200
            1200.0,   # > 1200
            5000.0,
        ]
        
        cmap_elev = ListedColormap(colours)
        norm = BoundaryNorm(bounds, len(colours), clip=True)
        
        # Plot as semi‑transparent background
        elev_plot = ax.pcolormesh(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            cmap=cmap_elev,
            norm=norm,
            alpha=1.0,
            transform=ccrs.PlateCarree(),
            # zorder=2,
        )

        # Draw 0 m contour as an accurate coastline
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[0.0],
            colors='black',
            linewidths=0.5,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        # Draw 400 m contour
        coast_contour = ax.contour(
            dem_lon_2d,
            dem_lat_2d,
            dem_data,
            levels=[400.0],
            colors='black',
            linewidths=0.3,
            transform=ccrs.PlateCarree(),
            zorder=15,  # above radar and topo
        )

        # print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

    except Exception as e:
        print(f'Terrain shading failed: {e}')
        # print('Falling back to simple land/ocean shading')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION

    # apply the condtional mask to the variable array before plotting
    ValidVariableArray = xgrid[VarNameLong] #.where(ConditionGrid)
    PlottingArray = ValidVariableArray[0,alti,:,:]

    GridViewer = ax.pcolormesh(xgrid.lon+ LonShift, xgrid.lat, PlottingArray, 
                               cmap=VarColourBar , norm=VarColourBar_norm, transform=ccrs.PlateCarree())

    # # ── plot feature centres of mass for this timestep ────────────────────────
    # result_timestamps = list(results.keys())
    # t_idx             = MinOfDay // 5

    # if t_idx < len(result_timestamps):
    #     df_this_t = results[result_timestamps[t_idx]]
    #     if not df_this_t.empty:
    #         ax.scatter(
    #             df_this_t['centre_lon'].values,
    #             df_this_t['centre_lat'].values,
    #             s         = 40,
    #             color     = 'black',
    #             transform = ccrs.PlateCarree(),
    #             zorder    = 25,
    #         )

    cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # tick every 10 dBZ

    # plot a star for the location of the radar on the map
    ax.plot(float(xgrid.radar_longitude) + LonShift, float(xgrid.radar_latitude),
            marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
    ax.plot(float(xgrid.radar_longitude) + LonShift, float(xgrid.radar_latitude),
            marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)

    # Add MINOR gridlines (tenth degrees) - thin
    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
    gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

    # Add MID LEVEL gridlines (half degrees) - standard width with labels
    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
    gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

    # Add MAJOR gridlines (full degrees) - thick with labels
    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    
    # Format labels
    gl_mid.xformatter = LONGITUDE_FORMATTER
    gl_mid.yformatter = LATITUDE_FORMATTER
    
    gl_major.xformatter = LONGITUDE_FORMATTER
    gl_major.yformatter = LATITUDE_FORMATTER
    
    # Remove labels from top and right
    gl_mid.top_labels = False
    gl_mid.right_labels = False
    gl_mid.bottom_labels = True
    gl_mid.left_labels = True
    
    gl_major.top_labels = False
    gl_major.right_labels = False
    gl_major.bottom_labels = True
    gl_major.left_labels = True

    # # Add coastlines ON TOP of radar data
    # ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=13)
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    plt.title(VarName + ' (ρHV > ' + str(MinValidRhoHV) + ') for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')

    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/' + \
                                                                                           VarName + '/RhoHVlim' + str(int(MinValidRhoHV*100)) + 'Compressed/'
                                                                                                      # RhoHV 0.85 becomes 85 in file name
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
                 PlotType + str(Altitude) + 'm.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('doing')
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    # plt.close()

In [ ]:
# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 41 is Willis Island, 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '22' 

Altitude = 2000  # [m] choose a multiple of 500 m to look at a CAPI for

# CHOOSE YOUR VARIABLE
Var = 'Z'

In [ ]:
PlotType = 'Horz'
# RadarIDno = '22'
RadarFileDate = '20241205'
VarName = 'Reflectivity'
MinValidRhoHV = 0.00
Altitude = 2000 # [m]

In [ ]:
# GIF MAKER
# FOR Horizontal Cross Sections

for RadarIDno in ['106','24','23','8','66']:

    SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/' + \
                                                                                           VarName + '/RhoHVlim' + str(int(MinValidRhoHV*100)) + 'Compressed/'
                                                                                                      # RhoHV 0.85 becomes 85 in file name         
    # LOADING IMAGES
    files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named
    
    images = [Image.open(os.path.join(SavedFolder, f))
        for f in files
        if f.endswith(PlotType + str(Altitude) + 'm.png')]
    
    # DOWNSCALE IMAGES BEFORE MAKING GIF
    InverseScaleFactor = 3
    ScaleFactor = 1 / InverseScaleFactor
    # 0.5 = half resolution, 0.25 = quarter resolution etc.
    
    ResizedImages = []
    for img in images:
        NewWidth  = int(img.width  * ScaleFactor)
        NewHeight = int(img.height * ScaleFactor)
        ResizedImages.append(img.resize((NewWidth, NewHeight), Image.LANCZOS))
    
    GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Horz/' + RadarIDno + '/' + RadarFileDate + '/'
    GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarName + '_RhoHVlim' + str(int(MinValidRhoHV*100)) + '_' + str(Altitude) + 'm.gif'
    
    GIFsavePath = GIFsaveFolder + GIFsaveFile
    
    # make a folder to store the new GIF in if one does not exist already
    if not Path(GIFsaveFolder).exists():
        Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)
    
    # Save as looping GIF
    ResizedImages[0].save(GIFsavePath, save_all=True, append_images=ResizedImages[1:], duration=200, loop=0)          
                                                                       # ms per frame       0 = loop forever
    print('Saved GIF for ' + RadarFileDate)

In [ ]:
# PLOT OF RADAR LOCATIONS ACROSS QUEENSLAND

CoarsenLevel = 4  # increase this to go faster, decrease for more detail

# list of all of the ID numbers of the radars you want on the map
RadarIDnoList = [41, 19, 106, 24, 22, 23, 8, 66, 78, 74, 72, 98, 108, 50]
RadarIDnoListStrs = [str(x) for x in RadarIDnoList]

FavouriteRadarsList = [41, 19, 106, 24, 22, 23, 8, 66] # North to South
FavouriteRadarsListStrs = [str(x) for x in FavouriteRadarsList]
# Others North to South: [78, 74, 72, 98, 108, 50]

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald



fig, ax = plt.subplots(figsize=(8, 6), 
                       subplot_kw={'projection': ccrs.PlateCarree()})


# TERRAIN SHADING USING LOCAL GEBCO DEM
lon_min, lon_max = 140.5, 154.7
lat_min, lat_max = -29.2, -10.5

try:
    gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    
    # print(f"Attempting to load DEM with bounds:")
    # print(f"  Lon: {lon_min} to {lon_max}")
    # print(f"  Lat: {lat_min} to {lat_max}")
    
    dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)
    
    if dem_da is None:
        raise ValueError('GEBCO DEM returned None')

    # Coarsen the DEM for faster plotting (every Nth point)
    dem_da = dem_da.coarsen(lon=CoarsenLevel, lat=CoarsenLevel, boundary='trim').mean()

    
    dem_lon = dem_da.lon.values
    dem_lat = dem_da.lat.values
    dem_data = dem_da.values

    # print(f"DEM data range: {np.nanmin(dem_data)} to {np.nanmax(dem_data)} m")
    # print(f"NaN count: {np.isnan(dem_data).sum()}")

    # Make 2D lon/lat grids if necessary
    if dem_lon.ndim == 1 and dem_lat.ndim == 1:
        dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
    else:
        dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

    # Ensure we have some valid data
    valid = np.isfinite(dem_data)
    if not np.any(valid):
        raise ValueError('DEM has no finite values in this domain')

    colours = [
        '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
    
        '#c4dec2',  # 1: 0–200 m, pale green
        '#e4edc9',  # 2: 200–400 m, greenish-yellow
        '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
        '#e9d7bd',  # 4: 600–800 m, light tan
        '#ddc4aa',  # 5: 800–1000 m, tan
        '#cfb194',  # 6: 1000–1200 m, light brown
        '#b58f6e',  # 7: > 1200 m, darker brown
    ]
    
    bounds = [
        -1000.0,  # ocean below 0
        0.0,      # 0–200
        200.0,    # 200–400
        400.0,    # 400–600
        600.0,    # 600–800
        800.0,    # 800–1000
        1000.0,   # 1000–1200
        1200.0,   # > 1200
        5000.0,
    ]
    
    cmap_elev = ListedColormap(colours)
    norm = BoundaryNorm(bounds, len(colours), clip=True)
    
    # Plot as semi‑transparent background
    elev_plot = ax.pcolormesh(
        dem_lon_2d,
        dem_lat_2d,
        dem_data,
        cmap=cmap_elev,
        norm=norm,
        alpha=0.8,
        transform=ccrs.PlateCarree(),
        # zorder=2,
    )

    # Draw 0 m contour as an accurate coastline
    coast_contour = ax.contour(
        dem_lon_2d,
        dem_lat_2d,
        dem_data,
        levels=[0.0],
        colors='black',
        linewidths=0.5,
        transform=ccrs.PlateCarree(),
        zorder=15,  # above radar and topo
    )

    # # Draw 400 m contour
    # coast_contour = ax.contour(
    #     dem_lon_2d,
    #     dem_lat_2d,
    #     dem_data,
    #     levels=[400.0],
    #     colors='black',
    #     linewidths=0.3,
    #     transform=ccrs.PlateCarree(),
    #     zorder=15,  # above radar and topo
    # )

    # print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

except Exception as e:
    print(f'Terrain shading failed: {e}')
    # print('Falling back to simple land/ocean shading')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.30, zorder=1)
    ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.30, zorder=2)

ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

# Add MINOR gridlines (degrees) - thin
gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
gl_minor.xlocator = mticker.MultipleLocator(1.0)  # Every 1 degree
gl_minor.ylocator = mticker.MultipleLocator(1.0)  # Every 1 degree

# Add MID LEVEL gridlines (five degrees) - standard width with labels
gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
gl_mid.xlocator = mticker.MultipleLocator(5.0)  # Every 5 degrees
gl_mid.ylocator = mticker.MultipleLocator(5.0)  # Every 5 degrees

# Add MAJOR gridlines (ten degrees) - thick with labels
gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
gl_major.xlocator = mticker.MultipleLocator(10.0)  # Every 10 degrees
gl_major.ylocator = mticker.MultipleLocator(10.0)  # Every 10 degrees

# Format labels
gl_mid.xformatter = LONGITUDE_FORMATTER
gl_mid.yformatter = LATITUDE_FORMATTER

gl_major.xformatter = LONGITUDE_FORMATTER
gl_major.yformatter = LATITUDE_FORMATTER

# Remove labels from top and right
gl_mid.top_labels = False
gl_mid.right_labels = False
gl_mid.bottom_labels = True
gl_mid.left_labels = True

gl_major.top_labels = False
gl_major.right_labels = False
gl_major.bottom_labels = True
gl_major.left_labels = True

# JUST CHOOSING AN EXAMPLE RADAR FILE FROM 2024-12-05T12:00:00Z FOR EACH RADAR TO EXTRACT SITE COORDINATES
RadarGridsFolder = 'CompressedRadarGrids'
YYYY = '2024'
MM = '12'
DD= '05'
RadarFileDate = YYYY + MM + DD
RadarFileTime = '120000'

for RadarIDno in RadarIDnoListStrs:

    if RadarIDno in FavouriteRadarsListStrs:
        IsFave = 1
    else:
        IsFave = 0

    # RADAR CHOICE FOLLOW-ON
    if (RadarIDno == '22'):
        RadarSiteName = 'Mackay'
    elif (RadarIDno == '106'):
        RadarSiteName = 'Townsville'
    elif (RadarIDno == '66'):
        RadarSiteName = 'Mt Staplyton (Brisbane)'
    elif (RadarIDno == '50'):
        RadarSiteName = 'Marburg'
    elif (RadarIDno == '19'):
        RadarSiteName = 'Cairns'
    elif (RadarIDno == '8'):
        RadarSiteName = 'Gympie'
    elif (RadarIDno == '24'):
        RadarSiteName = 'Bowen'
    elif (RadarIDno == '23'):
        RadarSiteName = 'Gladstone'
    elif (RadarIDno == '74'):
        RadarSiteName = 'Greenvale'
    elif (RadarIDno == '98'):
        RadarSiteName = 'Taroom'
    elif (RadarIDno == '108'):
        RadarSiteName = 'Towoomba'
    elif (RadarIDno == '78'):
        RadarSiteName = 'Weipa'
    elif (RadarIDno == '72'):
        RadarSiteName = 'Emerald'
    elif (RadarIDno == '41'):
        RadarSiteName = 'Willis Island'
    else:
        RadarSiteName = 'Site ' + RadarIDno
    
    # TOWNSVILLE LON NEEDS TO BE SHIFTED
    # TOWNSVILLE LON NEEDS TO BE SHIFTED
    # Longitude shift correction for known coordinate errors
    if RadarSiteName == 'Townsville':
        LonShift = 146.5505 - (-19.4195)
    else:
        LonShift = 0.0
        
    RadarGridsFolder = 'CompressedRadarGrids'
    NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                                 + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
    # try to load in the netcdf file and if it doesn't work, just keep going through the loop

    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # collect booleans for if this radar data has Z, V, and or ZDR as variables
    HasZ = 'corrected_reflectivity' in xgrid
    HasV = 'corrected_velocity' in xgrid
    HasDP = 'corrected_differential_reflectivity' in xgrid

    # collect radar location
    radar_lon = float(xgrid.radar_longitude) + LonShift
    radar_lat = float(xgrid.radar_latitude)

    if HasZ and HasV and HasDP:
        RadarColour = [0.50, 0.00, 1.00]
    elif HasZ and HasV and ~HasDP:
        RadarColour = [0.00, 0.00, 0.75]
    elif HasZ and ~HasV and ~HasDP:
        RadarColour = [0.00, 0.60, 0.00]
    else:
        RadarColour = [0.00, 0.00, 0.00]  

    # make the favourite (coastal) radars more noticable
    RingAlpha = 0.12 if IsFave else 0.05
    StarAlpha = 1.00 if IsFave else 0.25

    # plot a star for the location of the radar on the map
    ax.plot(radar_lon, radar_lat, marker='*', color=RadarColour, markersize=6, alpha=StarAlpha, transform=ccrs.PlateCarree(), zorder=20)
    ax.plot(radar_lon, radar_lat, marker='*', color='white',     markersize=2, alpha=StarAlpha, transform=ccrs.PlateCarree(), zorder=21)

    # label the radar sites (move the labels up if they are close to Brisbane)
    if (RadarSiteName == 'Marburg'):
        LabelDisplacementLon = -0.2
        LabelDisplacementLat = -0.05
    elif (RadarSiteName == 'Towoomba'):
        LabelDisplacementLon = -0.2
        LabelDisplacementLat = -0.075
    elif (RadarSiteName == 'Mt Staplyton (Brisbane)'):
        LabelDisplacementLon = -0.2
        LabelDisplacementLat = -0.25
    elif (RadarSiteName == 'Weipa'):
        LabelDisplacementLon = -0.4
        LabelDisplacementLat = -0.1
    elif (RadarSiteName == 'Bowen'):
        LabelDisplacementLon = -0.2
        LabelDisplacementLat = -0.4
    else:
        LabelDisplacementLon = -0.24
        LabelDisplacementLat = -0.24
    ax.text(radar_lon + LabelDisplacementLon, radar_lat + LabelDisplacementLat, RadarSiteName,
        fontsize=5, color=RadarColour, alpha = StarAlpha, ha='right', transform=ccrs.PlateCarree(), zorder=25)


    # add a filled range ring around the radar
    # RADAR RINGS DO NOT TAKE INTO ACCOUNT HOW LONGITUDE DEGREES ARE SMALLER THAN LATITUDE DEGREES AWAY FROM THE EQUATOR
    circle = Circle(xy=(radar_lon, radar_lat),
        radius=150/111.0,  # convert km to degrees (approx)
        facecolor=RadarColour,
        edgecolor=RadarColour,
        linewidth=0.5,
        alpha=RingAlpha,
        transform=ccrs.PlateCarree(),
        zorder=5)
    ax.add_patch(circle)

    legend_elements = [
        Line2D([0], [0], marker='*', color='none', markerfacecolor=[0.50, 0.00, 1.00], markeredgewidth=0, markersize=10, label='Dual-Pol Doppler'),
        Line2D([0], [0], marker='*', color='none', markerfacecolor=[0.00, 0.00, 0.75], markeredgewidth=0, markersize=10, label='Single-Pol Doppler'),
        Line2D([0], [0], marker='*', color='none', markerfacecolor=[0.00, 0.60, 0.00], markeredgewidth=0, markersize=10, label='Single-Pol Non-Doppler'),
    ]
        
    legend = ax.legend(
        handles=legend_elements,
        loc='upper right',
        framealpha=1.0,
        facecolor='#f8f8f8',
        edgecolor='#cccccc',
        fontsize=10,
    )
    legend.set_zorder(50)

    plt.title('Queensland Coast Radar Locations\nAnd Coverage (150 km Range Rings)')

    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
    
    SaveFile   = 'QueenslandCoastRadarLocationsCoverage.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('Creating Folder: ' + SaveFolder)
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    # plt.close()